# 📊 PathOptLearn — Progress Tracking Dashboard

This notebook implements the **Progress Tracking** system for PathOptLearn.

**Per-student tracking:**
1. Video ID + title watched
2. Score per quiz (raw + percentage)
3. Passed or not
4. Evaluation text + recommendation received
5. Timestamp
6. Videos already watched (to avoid repeats)
7. Learning path so far (sequence of videos in order)

In [1]:
# ── CELL 1: Install Dependencies ───────────────────────────
!pip install -q fastapi uvicorn pyngrok nest-asyncio matplotlib

In [2]:
# ── CELL 2: Database Setup ────────────────────────────────
import sqlite3
import json
import os
from datetime import datetime

DB_PATH = "pathoptlearn_progress.db"


def get_db():
    """Get a database connection with row factory."""
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn


def init_db():
    """Initialize the database with the student_progress table."""
    conn = get_db()
    cursor = conn.cursor()

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS student_progress (
            id              INTEGER PRIMARY KEY AUTOINCREMENT,
            student_id      TEXT NOT NULL,
            video_id        TEXT NOT NULL,
            video_title     TEXT NOT NULL,
            score_raw       INTEGER NOT NULL,
            score_total     INTEGER NOT NULL,
            score_pct       REAL NOT NULL,
            passed          BOOLEAN NOT NULL,
            evaluation_text TEXT DEFAULT '',
            recommendation  TEXT DEFAULT '',
            timestamp       TEXT NOT NULL,
            topic           TEXT DEFAULT '',
            weak_areas      TEXT DEFAULT '[]'
        )
    """)

    # Index for fast lookups by student
    cursor.execute("""
        CREATE INDEX IF NOT EXISTS idx_student_id
        ON student_progress(student_id)
    """)

    conn.commit()
    conn.close()
    print(f"✅ Database initialized at: {os.path.abspath(DB_PATH)}")


init_db()

✅ Database initialized at: /home/moujar/dev/PathOptLearn/pathoptlearn_progress.db


In [3]:
# ── CELL 3: CRUD Functions ────────────────────────────────

def save_session(
    student_id: str,
    video_id: str,
    video_title: str,
    score_raw: int,
    score_total: int,
    passed: bool,
    evaluation_text: str = "",
    recommendation: str = "",
    topic: str = "",
    weak_areas: list = None,
) -> int:
    """
    Save a completed learning session to the database.
    Returns the session ID.
    """
    if weak_areas is None:
        weak_areas = []

    score_pct = (score_raw / score_total * 100) if score_total > 0 else 0
    timestamp = datetime.utcnow().isoformat() + "Z"

    conn = get_db()
    cursor = conn.cursor()
    cursor.execute("""
        INSERT INTO student_progress
        (student_id, video_id, video_title, score_raw, score_total, score_pct,
         passed, evaluation_text, recommendation, timestamp, topic, weak_areas)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        student_id, video_id, video_title, score_raw, score_total, score_pct,
        passed, evaluation_text, recommendation, timestamp, topic,
        json.dumps(weak_areas),
    ))

    session_id = cursor.lastrowid
    conn.commit()
    conn.close()

    print(f"  💾 Saved session #{session_id} for student '{student_id}'")
    return session_id


def get_student_history(student_id: str) -> list:
    """
    Get full learning history for a student, ordered by timestamp.
    """
    conn = get_db()
    cursor = conn.cursor()
    cursor.execute("""
        SELECT * FROM student_progress
        WHERE student_id = ?
        ORDER BY timestamp ASC
    """, (student_id,))

    rows = cursor.fetchall()
    conn.close()

    history = []
    for row in rows:
        history.append({
            "id":              row["id"],
            "video_id":        row["video_id"],
            "video_title":     row["video_title"],
            "score_raw":       row["score_raw"],
            "score_total":     row["score_total"],
            "score_pct":       row["score_pct"],
            "passed":          bool(row["passed"]),
            "evaluation_text": row["evaluation_text"],
            "recommendation":  row["recommendation"],
            "timestamp":       row["timestamp"],
            "topic":           row["topic"],
            "weak_areas":      json.loads(row["weak_areas"]) if row["weak_areas"] else [],
        })

    return history


def get_watched_videos(student_id: str) -> list:
    """
    Get list of video IDs the student has already watched.
    Used by the recommender to avoid repeats.
    """
    conn = get_db()
    cursor = conn.cursor()
    cursor.execute("""
        SELECT DISTINCT video_id FROM student_progress
        WHERE student_id = ?
    """, (student_id,))

    video_ids = [row["video_id"] for row in cursor.fetchall()]
    conn.close()
    return video_ids


def get_learning_path(student_id: str) -> list:
    """
    Get the ordered sequence of videos the student watched.
    Returns list of {video_id, video_title, score_pct, passed, timestamp}.
    """
    conn = get_db()
    cursor = conn.cursor()
    cursor.execute("""
        SELECT video_id, video_title, score_pct, passed, timestamp
        FROM student_progress
        WHERE student_id = ?
        ORDER BY timestamp ASC
    """, (student_id,))

    path = []
    for row in cursor.fetchall():
        path.append({
            "video_id":    row["video_id"],
            "video_title": row["video_title"],
            "score_pct":   row["score_pct"],
            "passed":      bool(row["passed"]),
            "timestamp":   row["timestamp"],
        })

    conn.close()
    return path


def get_dashboard_stats(student_id: str) -> dict:
    """
    Get summary statistics for the student dashboard.
    """
    history = get_student_history(student_id)

    if not history:
        return {
            "total_sessions":   0,
            "total_videos":     0,
            "avg_score_pct":    0,
            "pass_rate":        0,
            "total_passed":     0,
            "total_failed":     0,
            "score_history":    [],
            "learning_path":    [],
            "all_weak_areas":   [],
        }

    total = len(history)
    passed = sum(1 for h in history if h["passed"])
    avg_score = sum(h["score_pct"] for h in history) / total
    unique_videos = len(set(h["video_id"] for h in history))

    # Collect all weak areas across sessions
    all_weak = []
    for h in history:
        all_weak.extend(h.get("weak_areas", []))
    # Count frequency
    from collections import Counter
    weak_counts = Counter(all_weak).most_common(10)

    return {
        "total_sessions":   total,
        "total_videos":     unique_videos,
        "avg_score_pct":    round(avg_score, 1),
        "pass_rate":        round(passed / total * 100, 1),
        "total_passed":     passed,
        "total_failed":     total - passed,
        "score_history":    [{"score_pct": h["score_pct"], "timestamp": h["timestamp"]} for h in history],
        "learning_path":    get_learning_path(student_id),
        "all_weak_areas":   [{'area': w, 'count': c} for w, c in weak_counts],
    }


print("✅ CRUD functions ready!")

✅ CRUD functions ready!


In [4]:
# ── CELL 4: Dashboard Visualization ──────────────────────
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend for Colab
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime
from IPython.display import display, HTML


def plot_score_progression(student_id: str):
    """
    Plot score progression over time as a line chart.
    """
    history = get_student_history(student_id)
    if not history:
        print("No data to plot.")
        return

    dates = [datetime.fromisoformat(h['timestamp'].replace('Z', '+00:00')) for h in history]
    scores = [h['score_pct'] for h in history]
    passed = [h['passed'] for h in history]

    fig, ax = plt.subplots(figsize=(12, 5))

    # Plot line
    ax.plot(range(len(scores)), scores, 'b-o', linewidth=2, markersize=8, label='Score %')

    # Color markers by pass/fail
    for i, (score, did_pass) in enumerate(zip(scores, passed)):
        color = '#2ecc71' if did_pass else '#e74c3c'
        ax.plot(i, score, 'o', color=color, markersize=10, zorder=5)

    # Pass threshold line
    ax.axhline(y=60, color='orange', linestyle='--', alpha=0.7, label='Pass threshold (60%)')

    ax.set_xlabel('Quiz #', fontsize=12)
    ax.set_ylabel('Score (%)', fontsize=12)
    ax.set_title(f'Score Progression — Student: {student_id}', fontsize=14, fontweight='bold')
    ax.set_ylim(0, 105)
    ax.legend()
    ax.grid(True, alpha=0.3)

    # Add video titles as x-tick labels
    video_labels = [h['video_title'][:20] + '...' if len(h['video_title']) > 20 else h['video_title'] for h in history]
    ax.set_xticks(range(len(scores)))
    ax.set_xticklabels(video_labels, rotation=45, ha='right', fontsize=9)

    plt.tight_layout()
    plt.savefig('score_progression.png', dpi=100, bbox_inches='tight')
    plt.show()


def plot_learning_path(student_id: str):
    """
    Display the learning path as a visual timeline.
    """
    path = get_learning_path(student_id)
    if not path:
        print("No learning path data.")
        return

    fig, ax = plt.subplots(figsize=(14, 3))

    for i, step in enumerate(path):
        color = '#2ecc71' if step['passed'] else '#e74c3c'
        ax.scatter(i, 0, s=300, c=color, zorder=5, edgecolors='white', linewidth=2)

        # Title below
        title = step['video_title'][:25] + '...' if len(step['video_title']) > 25 else step['video_title']
        ax.annotate(title, (i, 0), textcoords="offset points",
                    xytext=(0, -25), ha='center', fontsize=8, rotation=30)

        # Score above
        ax.annotate(f"{step['score_pct']:.0f}%", (i, 0), textcoords="offset points",
                    xytext=(0, 15), ha='center', fontsize=10, fontweight='bold')

        # Arrow to next
        if i < len(path) - 1:
            ax.annotate('', xy=(i + 0.8, 0), xytext=(i + 0.2, 0),
                        arrowprops=dict(arrowstyle='->', color='gray', lw=2))

    ax.set_xlim(-0.5, len(path) - 0.5)
    ax.set_ylim(-0.8, 0.5)
    ax.set_title(f'Learning Path — Student: {student_id}', fontsize=14, fontweight='bold')
    ax.axis('off')

    # Legend
    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], marker='o', color='w', markerfacecolor='#2ecc71', markersize=12, label='Passed'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor='#e74c3c', markersize=12, label='Failed'),
    ]
    ax.legend(handles=legend_elements, loc='upper right')

    plt.tight_layout()
    plt.savefig('learning_path.png', dpi=100, bbox_inches='tight')
    plt.show()


def display_dashboard(student_id: str):
    """
    Display full dashboard with stats, charts, and learning path.
    """
    stats = get_dashboard_stats(student_id)

    if stats['total_sessions'] == 0:
        print(f"No data for student '{student_id}'.")
        return stats

    print("=" * 60)
    print(f"  📊 DASHBOARD — Student: {student_id}")
    print("=" * 60)
    print(f"  📹 Videos watched:  {stats['total_videos']}")
    print(f"  📝 Total quizzes:   {stats['total_sessions']}")
    print(f"  📈 Average score:   {stats['avg_score_pct']}%")
    print(f"  ✅ Pass rate:       {stats['pass_rate']}%")
    print(f"  ✅ Passed:          {stats['total_passed']}")
    print(f"  ❌ Failed:          {stats['total_failed']}")

    if stats['all_weak_areas']:
        print(f"\n  ⚠️ Top weak areas:")
        for wa in stats['all_weak_areas'][:5]:
            print(f"     - {wa['area']} (appeared {wa['count']}x)")

    print("=" * 60)

    # Plot charts
    plot_score_progression(student_id)
    plot_learning_path(student_id)

    return stats


print("✅ Dashboard visualization functions ready!")

✅ Dashboard visualization functions ready!


In [5]:
# ── CELL 5: FastAPI Endpoints ─────────────────────────────
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import Optional, List
import uvicorn
import nest_asyncio
from pyngrok import ngrok
import threading
import time

nest_asyncio.apply()

app = FastAPI(title="PathOptLearn Progress Tracking API")


class SaveSessionRequest(BaseModel):
    student_id:      str
    video_id:        str
    video_title:     str
    score_raw:       int
    score_total:     int
    passed:          bool
    evaluation_text: Optional[str] = ""
    recommendation:  Optional[str] = ""
    topic:           Optional[str] = ""
    weak_areas:      Optional[List[str]] = []


class SaveSessionResponse(BaseModel):
    session_id: int
    message:    str


@app.get("/health")
def health():
    return {"status": "ok", "service": "progress_tracking"}


@app.post("/progress/save", response_model=SaveSessionResponse)
def save_progress(req: SaveSessionRequest):
    """Save a completed learning session."""
    if not req.student_id.strip():
        raise HTTPException(status_code=400, detail="student_id is empty.")

    session_id = save_session(
        student_id=req.student_id,
        video_id=req.video_id,
        video_title=req.video_title,
        score_raw=req.score_raw,
        score_total=req.score_total,
        passed=req.passed,
        evaluation_text=req.evaluation_text,
        recommendation=req.recommendation,
        topic=req.topic,
        weak_areas=req.weak_areas,
    )

    return {"session_id": session_id, "message": "Session saved successfully."}


@app.get("/progress/{student_id}")
def get_progress(student_id: str):
    """Get full learning history for a student."""
    history = get_student_history(student_id)
    return {"student_id": student_id, "total_sessions": len(history), "history": history}


@app.get("/progress/{student_id}/watched")
def get_watched(student_id: str):
    """Get list of watched video IDs (for recommender integration)."""
    video_ids = get_watched_videos(student_id)
    return {"student_id": student_id, "watched_video_ids": video_ids}


@app.get("/progress/{student_id}/path")
def get_path(student_id: str):
    """Get ordered learning path."""
    path = get_learning_path(student_id)
    return {"student_id": student_id, "learning_path": path}


@app.get("/progress/{student_id}/dashboard")
def get_dashboard(student_id: str):
    """Get dashboard statistics."""
    stats = get_dashboard_stats(student_id)
    return {"student_id": student_id, **stats}


print("✅ FastAPI endpoints defined!")
print("Endpoints: /progress/save, /progress/{id}, /progress/{id}/watched, /progress/{id}/path, /progress/{id}/dashboard")

✅ FastAPI endpoints defined!
Endpoints: /progress/save, /progress/{id}, /progress/{id}/watched, /progress/{id}/path, /progress/{id}/dashboard


In [14]:

NGROK_AUTH_TOKEN = "3AIhGm2Ih8LkOrw7B44HG8NKU5S_7C516SgaGBYDDWxdALaaB"

ngrok.set_auth_token(NGROK_AUTH_TOKEN)
public_url = ngrok.connect(8002)

print("\n" + "=" * 60)
print(f"  ✅ Progress API is live at: {public_url}")
print(f"  Health check:     {public_url}/health")
print(f"  Save session:     POST {public_url}/progress/save")
print(f"  Get history:      GET  {public_url}/progress/{{student_id}}")
print(f"  Get watched:      GET  {public_url}/progress/{{student_id}}/watched")
print(f"  Get dashboard:    GET  {public_url}/progress/{{student_id}}/dashboard")
print("=" * 60)


def run_server():
    nest_asyncio.apply()
    uvicorn.run(app, host="0.0.0.0", port=8002)


print("\nStarting server in background thread...")
server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

print("Server started! This cell will block to keep it alive.")
print("Interrupt this cell to stop the server.\n")

try:
    while True:
        time.sleep(3600)
except KeyboardInterrupt:
    print("\nServer stopped.")

PyngrokNgrokHTTPError: ngrok client exception, API returned 502: {"error_code":103,"status_code":502,"msg":"failed to start tunnel","details":{"err":"failed to start tunnel: The endpoint 'https://dulotic-fumigatory-romona.ngrok-free.dev' is already online. Either\n1. stop your existing endpoint first, or\n2. start both endpoints with `--pooling-enabled` to load balance between them.\r\n\r\nERR_NGROK_334\r\n"}}


In [15]:
# ── CELL 7: Demo — Save Sample Data & Display Dashboard ──
# Run this cell INSTEAD of Cell 6 (server) to test locally.

print("📝 Saving sample learning sessions...\n")

# Simulate a student's learning journey
sample_sessions = [
    {
        "student_id": "demo_student_1",
        "video_id": "WUvTyaaNkzM",
        "video_title": "Introduction to Calculus",
        "score_raw": 3, "score_total": 5,
        "passed": True,
        "evaluation_text": "Good understanding of basics.",
        "recommendation": "Move on to derivatives.",
        "topic": "calculus basics",
        "weak_areas": ["limits"],
    },
    {
        "student_id": "demo_student_1",
        "video_id": "abc123",
        "video_title": "Derivatives Explained",
        "score_raw": 2, "score_total": 5,
        "passed": False,
        "evaluation_text": "Struggled with chain rule applications.",
        "recommendation": "Review chain rule with simpler examples.",
        "topic": "calculus derivatives",
        "weak_areas": ["chain rule", "implicit differentiation"],
    },
    {
        "student_id": "demo_student_1",
        "video_id": "def456",
        "video_title": "Chain Rule Made Easy",
        "score_raw": 4, "score_total": 5,
        "passed": True,
        "evaluation_text": "Excellent improvement on chain rule!",
        "recommendation": "Ready for integration.",
        "topic": "calculus derivatives",
        "weak_areas": [],
    },
    {
        "student_id": "demo_student_1",
        "video_id": "ghi789",
        "video_title": "Introduction to Integration",
        "score_raw": 3, "score_total": 5,
        "passed": True,
        "evaluation_text": "Good grasp of basic integrals.",
        "recommendation": "Practice integration by substitution.",
        "topic": "calculus integration",
        "weak_areas": ["substitution method"],
    },
]

for session in sample_sessions:
    save_session(**session)

print("\n" + "=" * 60)
print("\n📊 Displaying dashboard for demo_student_1:\n")
stats = display_dashboard("demo_student_1")

print("\n📋 Watched videos list:")
watched = get_watched_videos("demo_student_1")
for vid in watched:
    print(f"  - {vid}")

📝 Saving sample learning sessions...

  💾 Saved session #5 for student 'demo_student_1'
  💾 Saved session #6 for student 'demo_student_1'
  💾 Saved session #7 for student 'demo_student_1'
  💾 Saved session #8 for student 'demo_student_1'


📊 Displaying dashboard for demo_student_1:

  📊 DASHBOARD — Student: demo_student_1
  📹 Videos watched:  4
  📝 Total quizzes:   8
  📈 Average score:   60.0%
  ✅ Pass rate:       75.0%
  ✅ Passed:          6
  ❌ Failed:          2

  ⚠️ Top weak areas:
     - limits (appeared 2x)
     - chain rule (appeared 2x)
     - implicit differentiation (appeared 2x)
     - substitution method (appeared 2x)

📋 Watched videos list:
  - WUvTyaaNkzM
  - abc123
  - def456
  - ghi789


/tmp/ipykernel_43729/1464350796.py:50: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_43729/1464350796.py:97: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
